# Notebook 04 â€” Privacy Filter (Face + Text Detection)

Pretrained models â€” **no training needed**. Sets up, tests, and exports to ONNX for Jetson.

## Cell 04-01 | Mount & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, torch, cv2
import numpy as np
from PIL import Image

BASE    = '/content/drive/MyDrive/ARGUS'
MODELS  = f'{BASE}/models'
EXPORTS = f'{BASE}/exports'
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'insightface', 'onnxruntime-gpu', 'opencv-python-headless'])
print(f'Ready | Device: {DEVICE}')

## Cell 04-02 | Download SCRFD Face Detector

In [ ]:
import subprocess, os, zipfile, tarfile as _tarfile

def _safe_dl(url, dest, min_mb=None):
    """wget -c resume. If file exists but too small, delete and restart."""
    if os.path.exists(dest):
        mb = os.path.getsize(dest) / 1e6
        if min_mb and mb < min_mb * 0.90:
            print(f"  Partial file {os.path.basename(dest)} ({mb:.0f}/{min_mb} MB) — removing")
            os.remove(dest)
        else:
            print(f"  ✓ {os.path.basename(dest)} ({mb:.0f} MB)")
            return True
    print(f"  Downloading {os.path.basename(dest)} ...")
    r = subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url])
    return r.returncode == 0

def _verify(path):
    """Return True if zip/tar is intact, delete it and return False if corrupt."""
    try:
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad: raise ValueError(bad)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with _tarfile.open(path) as t:
                t.getmembers()
        return True
    except Exception as e:
        print(f"  Corrupt archive ({e}) — removing for clean re-download")
        os.remove(path)
        return False

def _extract(path, dest):
    print(f"  Extracting {os.path.basename(path)} ...")
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    else:
        subprocess.run(["tar", "-xf", path, "-C", dest], check=True)


from huggingface_hub import hf_hub_download
import shutil

PRIVACY_DIR = f'{MODELS}/privacy'
BUFFALO_DIR = f'{PRIVACY_DIR}/buffalo_s'
DONE_FLAG   = f'{PRIVACY_DIR}/buffalo_done.flag'
os.makedirs(BUFFALO_DIR, exist_ok=True)

if os.path.exists(DONE_FLAG):
    print('✓ buffalo_s already downloaded')
else:
    # insightface downloads buffalo_s automatically on first prepare()
    # but we cache it to Drive so it survives Colab restarts.
    BUFFALO_URL = 'https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_s.zip'
    BUFFALO_ZIP = f'{PRIVACY_DIR}/buffalo_s.zip'
    ok = _safe_dl(BUFFALO_URL, BUFFALO_ZIP, min_mb=80)
    if not ok:
        print('⚠ Download failed — re-run to resume'); raise SystemExit
    if not _verify(BUFFALO_ZIP):
        print('Re-run to re-download corrupt archive'); raise SystemExit
    _extract(BUFFALO_ZIP, BUFFALO_DIR)
    open(DONE_FLAG, 'w').close()
    print('✅ buffalo_s ready')

# Tell insightface to use our cached copy
import insightface
face_app = insightface.app.FaceAnalysis(
    name='buffalo_s', root=PRIVACY_DIR,
    providers=['CUDAExecutionProvider','CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(320, 320))
print('✅ FaceAnalysis ready')

## Cell 04-03 | Test Face Detection & Blur

In [ ]:
import subprocess, os, zipfile, tarfile as _tarfile

def _safe_dl(url, dest, min_mb=None):
    """wget -c resume. If file exists but too small, delete and restart."""
    if os.path.exists(dest):
        mb = os.path.getsize(dest) / 1e6
        if min_mb and mb < min_mb * 0.90:
            print(f"  Partial file {os.path.basename(dest)} ({mb:.0f}/{min_mb} MB) — removing")
            os.remove(dest)
        else:
            print(f"  ✓ {os.path.basename(dest)} ({mb:.0f} MB)")
            return True
    print(f"  Downloading {os.path.basename(dest)} ...")
    r = subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url])
    return r.returncode == 0

def _verify(path):
    """Return True if zip/tar is intact, delete it and return False if corrupt."""
    try:
        if path.endswith(".zip"):
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad: raise ValueError(bad)
        elif path.endswith((".tar.gz", ".tgz", ".tar")):
            with _tarfile.open(path) as t:
                t.getmembers()
        return True
    except Exception as e:
        print(f"  Corrupt archive ({e}) — removing for clean re-download")
        os.remove(path)
        return False

def _extract(path, dest):
    print(f"  Extracting {os.path.basename(path)} ...")
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(dest)
    else:
        subprocess.run(["tar", "-xf", path, "-C", dest], check=True)


import cv2, numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def blur_faces(image_bgr, face_app, blur_strength=51):
    # Detect faces and apply Gaussian blur — privacy-preserving
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    faces   = face_app.get(img_rgb)
    result  = image_bgr.copy()
    n_faces = len(faces)
    for face in faces:
        bbox = face.bbox.astype(int)
        x1, y1, x2, y2 = bbox[0], bbox[1], bbox[2], bbox[3]
        pad = 20
        x1 = max(0, x1-pad); y1 = max(0, y1-pad)
        x2 = min(image_bgr.shape[1], x2+pad); y2 = min(image_bgr.shape[0], y2+pad)
        roi = result[y1:y2, x1:x2]
        if roi.size > 0:
            result[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (blur_strength, blur_strength), 0)
    return result, n_faces

test_url  = 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/640px-Gatto_europeo4.jpg'
test_path = '/tmp/test_image.jpg'
_safe_dl(test_url, test_path)  # -c resume, re-download if 0 bytes

img = cv2.imread(test_path)
blurred, n = blur_faces(img, face_app)
print(f'Detected {n} face(s)')
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(img,     cv2.COLOR_BGR2RGB)); axes[0].set_title('Original')
axes[1].imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)); axes[1].set_title('Privacy Protected')
plt.tight_layout(); plt.savefig(f'{BASE}/logs/privacy_test.png'); plt.show()
print('✅ Privacy filter working')

## Cell 04-04 | CRAFT Text Region Detection

In [ ]:
import subprocess, sys, os

# EasyOCR replaces CRAFT for text detection.
# Models auto-download on first use (~30 MB, cached to Drive).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'easyocr'])
import easyocr, cv2
import numpy as np
from PIL import Image

EASYOCR_DIR = f'{MODELS}/privacy/easyocr'
os.makedirs(EASYOCR_DIR, exist_ok=True)

print('Loading EasyOCR (downloads ~30 MB on first run) ...')
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available(),
                         model_storage_directory=EASYOCR_DIR,
                         download_enabled=True, verbose=False)
print('EasyOCR ready')

# ── Quick smoke test ──────────────────────────────────────────────────────────
test_path = '/tmp/test_text.jpg'
if not os.path.exists(test_path):
    import urllib.request
    urllib.request.urlretrieve(
        'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg',
        test_path)

img_bgr = cv2.imread(test_path)
results = reader.detect(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), slope_ths=0.3, height_ths=1.0)
boxes = results[0] if results and results[0] else []
print(f'Text regions detected: {len(boxes)}')

# Blur detected regions
result = img_bgr.copy()
for bbox in boxes:
    x_min, x_max, y_min, y_max = bbox
    roi = result[y_min:y_max, x_min:x_max]
    if roi.size > 0:
        result[y_min:y_max, x_min:x_max] = cv2.GaussianBlur(roi, (31, 31), 0)

Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)).save(f'{BASE}/exports/text_blur_test.jpg')
print(f'Text-blurred image saved: {BASE}/exports/text_blur_test.jpg')
print('EasyOCR text detection working.')
